# 📊 Data Analysis Agent
**Hongchao Hu & Niyati Gohil — CS539 Project**

This notebook runs the full Data Analysis Agent inside Google Colab and exposes it as a public web app — **no local installation required**.

### How to use
1. Click **Runtime → Run all** (or press `Ctrl+F9`)
2. Wait ~60 seconds for setup to finish
3. Click the public URL that appears at the bottom of the last cell
4. In the web UI, paste your Gemini API key (get a free one at [aistudio.google.com](https://aistudio.google.com))
5. Upload any CSV file and start analyzing!

> **Note:** The public URL changes every time you restart the notebook. Just re-run all cells to get a new URL.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
import sys

print("Installing dependencies...")
!pip install -q \
    google-generativeai \
    python-dotenv \
    pandas \
    numpy \
    scikit-learn \
    matplotlib \
    seaborn \
    fastapi \
    "uvicorn[standard]" \
    pydantic \
    python-multipart \
    httpx

print("✅ Dependencies installed")

In [ ]:
# ── Cell 2: Write project files ───────────────────────────────────────────────
# This cell reconstructs the project source tree directly inside Colab.
# No git clone or file upload needed.

import os
from pathlib import Path

ROOT = Path("/content/data_analysis_agent")
ROOT.mkdir(exist_ok=True)
os.chdir(ROOT)

for d in ["src/tools", "static/css", "static/js", "uploads", "artifacts", "outputs"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

# ── src/__init__.py ──
(ROOT / "src/__init__.py").write_text("")
(ROOT / "src/tools/__init__.py").write_text("")

# ── src/config.py ──
(ROOT / "src/config.py").write_text('''
"""Configuration settings for the Data Analysis Agent"""
import os

class Config:
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
    GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-1.5-flash")
    TEMPERATURE = float(os.getenv("TEMPERATURE", "0.7"))
    MAX_TOKENS = int(os.getenv("MAX_TOKENS", "2048"))
    MAX_ROWS_TO_DISPLAY = 10
    MAX_COLUMNS_TO_ANALYZE = 50
    CORRELATION_THRESHOLD = 0.5
    FIGURE_SIZE = (10, 6)
    DPI = 100
    OUTPUT_DIR = "outputs"
    SAVE_FIGURES = True
''')

print("✅ Project structure created")

In [ ]:
# ── Cell 3: Write tool source files ───────────────────────────────────────────

(ROOT / "src/tools/inspection.py").write_text(r'''
"""Dataset Inspection Tool"""
import pandas as pd
from typing import Dict, Any, Optional
import os

class DatasetInspectionTool:
    def __init__(self):
        self.df: Optional[pd.DataFrame] = None
        self.file_path: Optional[str] = None

    def load_dataset(self, file_path: str) -> Dict[str, Any]:
        try:
            self.file_path = file_path
            self.df = pd.read_csv(file_path)
            return {"success": True, "message": f"Loaded {file_path}", "basic_info": self.get_basic_info()}
        except Exception as e:
            return {"success": False, "message": str(e), "basic_info": None}

    def get_basic_info(self) -> Dict[str, Any]:
        if self.df is None:
            return {"error": "No dataset loaded"}
        return {
            "file_name": os.path.basename(self.file_path) if self.file_path else "Unknown",
            "num_rows": len(self.df),
            "num_columns": len(self.df.columns),
            "column_names": self.df.columns.tolist(),
            "data_types": self.df.dtypes.astype(str).to_dict(),
            "memory_usage_mb": self.df.memory_usage(deep=True).sum() / 1024**2
        }

    def get_missing_values_info(self) -> Dict[str, Any]:
        if self.df is None:
            return {"error": "No dataset loaded"}
        missing_counts = self.df.isnull().sum()
        missing_percentages = (missing_counts / len(self.df) * 100).round(2)
        missing_info = pd.DataFrame({"missing_count": missing_counts, "missing_percentage": missing_percentages})
        missing_info = missing_info[missing_info["missing_count"] > 0]
        return {
            "total_missing_values": int(missing_counts.sum()),
            "columns_with_missing": missing_info.to_dict("index"),
            "percentage_complete": round((1 - self.df.isnull().sum().sum() / self.df.size) * 100, 2)
        }

    def get_dataframe(self) -> Optional[pd.DataFrame]:
        return self.df
''')

(ROOT / "src/tools/statistics.py").write_text(r'''
"""Statistical Analysis Tool"""
import pandas as pd
import numpy as np
from typing import Dict, Any, List, Optional

class StatisticalAnalysisTool:
    def __init__(self, df=None):
        self.df = df

    def set_dataframe(self, df):
        self.df = df

    def get_descriptive_statistics(self, columns=None):
        if self.df is None:
            return {"error": "No dataframe set"}
        numeric_df = self.df.select_dtypes(include=[np.number])
        if columns:
            numeric_df = numeric_df[columns]
        if numeric_df.empty:
            return {"error": "No numeric columns found"}
        stats = numeric_df.describe().to_dict()
        additional = {}
        for col in numeric_df.columns:
            additional[col] = {
                "variance": float(numeric_df[col].var()),
                "skewness": float(numeric_df[col].skew()),
                "kurtosis": float(numeric_df[col].kurtosis())
            }
        return {"basic_statistics": stats, "additional_statistics": additional}

    def get_categorical_distribution(self, columns=None, top_n=10):
        if self.df is None:
            return {"error": "No dataframe set"}
        categorical_df = self.df.select_dtypes(include=["object", "category"])
        if columns:
            categorical_df = categorical_df[columns]
        if categorical_df.empty:
            return {"error": "No categorical columns found"}
        distributions = {}
        for col in categorical_df.columns:
            vc = self.df[col].value_counts()
            distributions[col] = {
                "unique_values": int(self.df[col].nunique()),
                "top_values": vc.head(top_n).to_dict(),
                "top_percentages": (vc.head(top_n) / len(self.df) * 100).round(2).to_dict()
            }
        return distributions

    def get_correlation_matrix(self, method="pearson", threshold=None):
        if self.df is None:
            return {"error": "No dataframe set"}
        numeric_df = self.df.select_dtypes(include=[np.number])
        if numeric_df.empty or len(numeric_df.columns) < 2:
            return {"error": "Need at least 2 numeric columns"}
        corr = numeric_df.corr(method=method)
        high = []
        for i in range(len(corr.columns)):
            for j in range(i + 1, len(corr.columns)):
                v = corr.iloc[i, j]
                if threshold is None or abs(v) >= threshold:
                    high.append({"variable_1": corr.columns[i], "variable_2": corr.columns[j], "correlation": round(float(v), 3)})
        high.sort(key=lambda x: abs(x["correlation"]), reverse=True)
        return {"correlation_matrix": corr.round(3).to_dict(), "high_correlations": high, "method": method}
''')

print("✅ Tool files written")

In [ ]:
# ── Cell 4: Write visualization tool ──────────────────────────────────────────

(ROOT / "src/tools/visualization.py").write_text(r'''
"""Visualization Tool"""
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")  # headless backend for server
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, Any, List, Optional
import os
from datetime import datetime

class VisualizationTool:
    def __init__(self, df=None, output_dir="outputs"):
        self.df = df
        self.output_dir = output_dir
        self.created_plots = []
        sns.set_style("whitegrid")
        plt.rcParams["figure.figsize"] = (10, 6)
        plt.rcParams["figure.dpi"] = 100
        os.makedirs(output_dir, exist_ok=True)

    def set_dataframe(self, df):
        self.df = df

    def _save_plot(self, fig, base_name):
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"{base_name}_{ts}.png"
        filepath = os.path.join(self.output_dir, filename)
        fig.savefig(filepath, dpi=100, bbox_inches="tight")
        self.created_plots.append(filepath)
        return filepath

    def create_distribution_grid(self, columns=None, save=True):
        if self.df is None:
            return {"error": "No dataframe set"}
        numeric_df = self.df.select_dtypes(include=[np.number])
        if columns:
            numeric_df = numeric_df[columns]
        if numeric_df.empty:
            return {"error": "No numeric columns"}
        n_cols = min(3, len(numeric_df.columns))
        n_rows = (len(numeric_df.columns) + n_cols - 1) // n_cols
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
        axes_flat = axes.flatten() if hasattr(axes, "flatten") else [axes]
        for idx, col in enumerate(numeric_df.columns):
            axes_flat[idx].hist(numeric_df[col].dropna(), bins=30, edgecolor="black", alpha=0.7)
            axes_flat[idx].set_title(f"Distribution of {col}")
            axes_flat[idx].set_xlabel(col)
            axes_flat[idx].set_ylabel("Frequency")
        for idx in range(len(numeric_df.columns), len(axes_flat)):
            axes_flat[idx].set_visible(False)
        plt.tight_layout()
        result = {"type": "distribution_grid", "columns": list(numeric_df.columns)}
        if save:
            result["file_path"] = self._save_plot(fig, "distribution_grid")
        plt.close(fig)
        return result

    def create_correlation_heatmap(self, columns=None, method="pearson", title=None, save=True):
        if self.df is None:
            return {"error": "No dataframe set"}
        numeric_df = self.df.select_dtypes(include=[np.number])
        if columns:
            numeric_df = numeric_df[columns]
        if numeric_df.empty or len(numeric_df.columns) < 2:
            return {"error": "Need at least 2 numeric columns"}
        corr = numeric_df.corr(method=method)
        fig, ax = plt.subplots(figsize=(12, 10))
        sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, ax=ax, cbar_kws={"shrink": 0.8})
        ax.set_title(title or f"Correlation Heatmap ({method})")
        plt.tight_layout()
        result = {"type": "correlation_heatmap", "method": method}
        if save:
            result["file_path"] = self._save_plot(fig, "correlation_heatmap")
        plt.close(fig)
        return result

    def create_box_plot(self, columns=None, title=None, save=True):
        if self.df is None:
            return {"error": "No dataframe set"}
        numeric_df = self.df.select_dtypes(include=[np.number])
        if columns:
            numeric_df = numeric_df[columns]
        if numeric_df.empty:
            return {"error": "No numeric columns"}
        fig, ax = plt.subplots(figsize=(12, 6))
        numeric_df.boxplot(ax=ax)
        ax.set_title(title or "Box Plot - Outlier Detection")
        ax.set_ylabel("Value")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        result = {"type": "box_plot", "columns": list(numeric_df.columns)}
        if save:
            result["file_path"] = self._save_plot(fig, "box_plot")
        plt.close(fig)
        return result

    def create_bar_chart(self, column, top_n=10, title=None, save=True):
        if self.df is None:
            return {"error": "No dataframe set"}
        if column not in self.df.columns:
            return {"error": f"Column {column} not found"}
        fig, ax = plt.subplots(figsize=(12, 6))
        vc = self.df[column].value_counts().head(top_n)
        bars = ax.bar(range(len(vc)), vc.values, alpha=0.7)
        ax.set_xticks(range(len(vc)))
        ax.set_xticklabels(vc.index, rotation=45, ha="right")
        ax.set_xlabel(column)
        ax.set_ylabel("Count")
        ax.set_title(title or f"Top {top_n} Categories in {column}")
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), str(int(bar.get_height())), ha="center", va="bottom")
        plt.tight_layout()
        result = {"type": "bar_chart", "column": column}
        if save:
            result["file_path"] = self._save_plot(fig, f"bar_chart_{column}")
        plt.close(fig)
        return result

    def get_created_plots(self):
        return self.created_plots
''')

print("✅ Visualization tool written")

In [ ]:
# ── Cell 5: Write agent ────────────────────────────────────────────────────────

(ROOT / "src/agent.py").write_text(r'''
"""Data Analysis Agent"""
import json
from typing import Dict, Any, Optional
import google.generativeai as genai

from .config import Config
from .tools.inspection import DatasetInspectionTool
from .tools.statistics import StatisticalAnalysisTool
from .tools.visualization import VisualizationTool


class DataAnalysisAgent:
    def __init__(self, api_key: Optional[str] = None):
        self.api_key = api_key or Config.GEMINI_API_KEY
        genai.configure(api_key=self.api_key)
        self.model = genai.GenerativeModel(Config.GEMINI_MODEL)
        self.inspection_tool = DatasetInspectionTool()
        self.statistics_tool = StatisticalAnalysisTool()
        self.visualization_tool = VisualizationTool()

    def analyze(self, file_path: str, user_prompt: str) -> Dict[str, Any]:
        results = {
            "user_prompt": user_prompt,
            "file_path": file_path,
            "steps": [],
            "visualizations": [],
            "summary": "",
            "success": True
        }
        try:
            load_result = self.inspection_tool.load_dataset(file_path)
            if not load_result["success"]:
                results["success"] = False
                results["error"] = load_result["message"]
                return results

            results["steps"].append({"step": "load_dataset", "result": load_result["basic_info"]})
            df = self.inspection_tool.get_dataframe()
            self.statistics_tool.set_dataframe(df)
            self.visualization_tool.set_dataframe(df)

            basic_info = load_result["basic_info"]
            missing_info = self.inspection_tool.get_missing_values_info()
            results["steps"].append({"step": "inspect_dataset", "basic_info": basic_info, "missing_values": missing_info})

            analysis_plan = self._create_analysis_plan(basic_info, user_prompt)
            results["analysis_plan"] = self._format_plan_for_display(analysis_plan)

            execution_results = self._execute_analysis_plan(analysis_plan)
            results["steps"].extend(execution_results["steps"])
            results["visualizations"].extend(execution_results["visualizations"])

            results["summary"] = self._generate_summary(user_prompt, basic_info, execution_results)
        except Exception as e:
            results["success"] = False
            results["error"] = str(e)
        return results

    def _create_analysis_plan(self, basic_info, user_prompt):
        prompt = f"""You are a data analysis assistant. Based on the dataset information and user request, create a detailed analysis plan.

Dataset Information:
- Rows: {basic_info["num_rows"]}, Columns: {basic_info["num_columns"]}
- Column names: {", ".join(basic_info["column_names"])}
- Data types: {json.dumps(basic_info["data_types"])}

User Request: {user_prompt}

Return ONLY a JSON object:
{{"statistical_analyses": ["list"], "visualizations": ["list"], "focus_areas": ["list"]}}"""
        try:
            response = self.model.generate_content(prompt)
            text = response.text.strip().lstrip("```json").lstrip("```").rstrip("`").strip()
            return json.loads(text)
        except Exception:
            return {
                "statistical_analyses": ["descriptive_statistics", "correlation_analysis"],
                "visualizations": ["distribution_plots", "correlation_heatmap"],
                "focus_areas": ["data quality", "variable relationships"]
            }

    def _format_plan_for_display(self, plan):
        steps = []
        for a in plan.get("statistical_analyses", []):
            steps.append({"tool": "Statistical Analysis", "purpose": a.replace("_", " ").title()})
        for v in plan.get("visualizations", []):
            steps.append({"tool": "Visualization", "purpose": v.replace("_", " ").title()})
        for f in plan.get("focus_areas", []):
            steps.append({"tool": "Analysis Focus", "purpose": f"Investigate {f}"})
        return steps

    def _execute_analysis_plan(self, plan):
        results = {"steps": [], "visualizations": []}

        for analysis in plan.get("statistical_analyses", []):
            a = analysis.lower()
            if "descriptive" in a or "statistics" in a or "summary" in a:
                r = self.statistics_tool.get_descriptive_statistics()
                results["steps"].append({"step": "descriptive_statistics", "result": r})
            if "correlation" in a:
                r = self.statistics_tool.get_correlation_matrix(threshold=0.3)
                results["steps"].append({"step": "correlation_analysis", "result": r})
            if "categorical" in a or "distribution" in a:
                r = self.statistics_tool.get_categorical_distribution()
                if "error" not in r:
                    results["steps"].append({"step": "categorical_distribution", "result": r})

        for viz in plan.get("visualizations", []):
            v = viz.lower()
            if "distribution" in v or "histogram" in v:
                r = self.visualization_tool.create_distribution_grid()
                if "error" not in r:
                    results["visualizations"].append(r)
            if "correlation" in v or "heatmap" in v:
                r = self.visualization_tool.create_correlation_heatmap()
                if "error" not in r:
                    results["visualizations"].append(r)
            if "box" in v or "outlier" in v:
                r = self.visualization_tool.create_box_plot()
                if "error" not in r:
                    results["visualizations"].append(r)
            if "bar" in v or "categor" in v:
                cat_cols = self.visualization_tool.df.select_dtypes(include=["object", "category"]).columns.tolist() if self.visualization_tool.df is not None else []
                for col in cat_cols[:2]:
                    r = self.visualization_tool.create_bar_chart(col)
                    if "error" not in r:
                        results["visualizations"].append(r)

        return results

    def _generate_summary(self, user_prompt, basic_info, execution_results):
        prompt = f"""You are a data scientist summarizing an EDA. Be specific, reference numbers.

Original question: {user_prompt}
Dataset: {basic_info["num_rows"]} rows, {basic_info["num_columns"]} columns — {', '.join(basic_info["column_names"])}

Analysis results (JSON):
{json.dumps(execution_results, indent=2, default=str)}

Write a structured summary with these sections using markdown:
## Dataset Description
## Key Findings
## Suggested Next Steps"""
        try:
            return self.model.generate_content(prompt).text
        except Exception:
            numeric = [c for c, t in basic_info["data_types"].items() if "int" in t or "float" in t]
            categorical = [c for c, t in basic_info["data_types"].items() if "object" in t]
            return f"""## Dataset Description\n{basic_info["num_rows"]} rows, {basic_info["num_columns"]} columns.\n\n## Key Findings\n- Numeric columns: {', '.join(numeric[:5])}\n- Categorical columns: {', '.join(categorical[:5])}\n- Generated {len(execution_results["visualizations"])} visualizations.\n\n## Suggested Next Steps\n1. Review visualizations for patterns\n2. Investigate correlations\n3. Check for outliers"""
''')

print("✅ Agent written")

In [ ]:
# ── Cell 6: Write FastAPI app ──────────────────────────────────────────────────

(ROOT / "app.py").write_text(r'''
"""FastAPI Web Application for Data Analysis Agent"""
from fastapi import FastAPI, HTTPException, UploadFile, File, Form, BackgroundTasks, Header
from fastapi.responses import JSONResponse, FileResponse
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel, Field
from typing import Optional, List, Dict, Any
import os, json, time, uuid, shutil
from pathlib import Path

from src.agent import DataAnalysisAgent
from src.config import Config

UPLOAD_DIR = Path("uploads")
ARTIFACTS_DIR = Path("artifacts")
OUTPUT_DIR = Path("outputs")
STATIC_DIR = Path("static")

for d in [UPLOAD_DIR, ARTIFACTS_DIR, OUTPUT_DIR, STATIC_DIR]:
    d.mkdir(exist_ok=True)


class AnalysisResponse(BaseModel):
    artifact_id: str
    query: str
    success: bool
    summary: str
    analysis_plan: List[Dict[str, Any]]
    steps_executed: int
    visualizations: List[str]
    evaluation: Optional[Dict[str, Any]] = None
    latency: float
    artifact_path: Optional[str] = None


def create_app() -> FastAPI:
    app = FastAPI(title="Data Analysis Agent API", version="1.0.0", docs_url="/docs")
    app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])
    if STATIC_DIR.exists():
        app.mount("/static", StaticFiles(directory=str(STATIC_DIR)), name="static")
    return app


app = create_app()


def _get_agent(api_key: Optional[str]) -> DataAnalysisAgent:
    """Create a per-request agent using the caller-supplied API key."""
    key = api_key or Config.GEMINI_API_KEY
    if not key:
        raise HTTPException(status_code=401, detail="Gemini API key is required. Pass it in the X-API-Key header.")
    return DataAnalysisAgent(api_key=key)


def save_artifact(analysis_id, results):
    path = ARTIFACTS_DIR / f"{analysis_id}.json"
    path.write_text(json.dumps(results, indent=2, default=str))
    return str(path)


def cleanup_old(directory: Path, max_age_hours=24):
    import time as _time
    now = _time.time()
    for p in directory.glob("*"):
        if p.is_file() and (now - p.stat().st_mtime) / 3600 > max_age_hours:
            p.unlink()


@app.get("/")
async def root():
    idx = STATIC_DIR / "index.html"
    if idx.exists():
        return FileResponse(idx)
    return {"service": "Data Analysis Agent API", "docs": "/docs"}


@app.get("/health")
async def health():
    return {"status": "healthy", "version": "1.0.0", "agent_initialized": True}


@app.post("/upload-analyze", response_model=AnalysisResponse)
async def upload_and_analyze(
    file: UploadFile = File(...),
    query: str = Form(...),
    background_tasks: BackgroundTasks = None,
    x_api_key: Optional[str] = Header(None)
):
    if not file.filename.endswith(".csv"):
        raise HTTPException(status_code=400, detail="Only CSV files are supported")

    agent = _get_agent(x_api_key)

    upload_id = str(uuid.uuid4())
    file_path = UPLOAD_DIR / f"{upload_id}_{file.filename}"
    try:
        with open(file_path, "wb") as buf:
            shutil.copyfileobj(file.file, buf)

        start = time.time()
        results = agent.analyze(str(file_path), query)
        latency = round(time.time() - start, 2)

        if not results.get("success"):
            raise HTTPException(status_code=500, detail=results.get("error", "Analysis failed"))

        analysis_id = str(uuid.uuid4())
        artifact_path = save_artifact(analysis_id, results)

        if background_tasks:
            background_tasks.add_task(cleanup_old, ARTIFACTS_DIR, 24)
            background_tasks.add_task(cleanup_old, OUTPUT_DIR, 24)

        viz_files = []
        for viz in results.get("visualizations", []):
            if isinstance(viz, dict) and "file_path" in viz:
                fp = viz["file_path"]
                viz_files.append(fp.split("/")[-1] if "/" in fp else fp.split("\\")[-1])
            elif isinstance(viz, str):
                viz_files.append(viz)

        return AnalysisResponse(
            artifact_id=analysis_id,
            query=query,
            success=True,
            summary=results["summary"],
            analysis_plan=results.get("analysis_plan", []),
            steps_executed=len(results.get("steps", [])),
            visualizations=viz_files,
            evaluation=None,
            latency=latency,
            artifact_path=artifact_path
        )
    except HTTPException:
        raise
    except Exception as e:
        if file_path.exists():
            file_path.unlink()
        raise HTTPException(status_code=500, detail=str(e))


@app.get("/artifact/{analysis_id}")
async def get_artifact(analysis_id: str):
    path = ARTIFACTS_DIR / f"{analysis_id}.json"
    if not path.exists():
        raise HTTPException(status_code=404, detail="Artifact not found")
    return FileResponse(path, media_type="application/json")


@app.get("/visualization/{filename}")
async def get_visualization(filename: str):
    # Security: prevent path traversal
    safe = Path(filename).name
    path = OUTPUT_DIR / safe
    if not path.exists():
        raise HTTPException(status_code=404, detail="Visualization not found")
    return FileResponse(path, media_type="image/png")


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
''')

print("✅ app.py written")

In [ ]:
# ── Cell 7: Write frontend files ──────────────────────────────────────────────

html = r"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Data Analysis Agent</title>
    <link rel="stylesheet" href="/static/css/styles.css">
</head>
<body>
<div class="container">
    <header class="header">
        <h1>&#128202; Data Analysis Agent</h1>
        <p class="subtitle">Drop your CSV file and get instant insights powered by AI</p>
        <div class="status-indicator">
            <span class="status-dot" id="statusDot"></span>
            <span id="statusText">Checking status...</span>
        </div>
    </header>
    <main class="main-content">
        <!-- API Key Section -->
        <section class="apikey-section" id="apikeySection">
            <div class="apikey-card">
                <h2>&#128273; Gemini API Key</h2>
                <p>Enter your free Google Gemini API key to get started.
                   Get one at <a href="https://aistudio.google.com" target="_blank">aistudio.google.com</a>.</p>
                <div class="apikey-input-row">
                    <input type="password" id="apiKeyInput" placeholder="AIza..." autocomplete="off">
                    <button class="btn-secondary" id="saveKeyBtn">Save &amp; Continue</button>
                </div>
                <p class="apikey-note">Your key is never stored — it is only held in memory for this session.</p>
            </div>
        </section>

        <!-- Upload Section -->
        <section class="upload-section" id="uploadSection" style="display:none;">
            <div class="upload-area" id="uploadArea">
                <div class="upload-icon">&#128193;</div>
                <h2>Drop your CSV file here</h2>
                <p>or click to browse</p>
                <input type="file" id="fileInput" accept=".csv" hidden>
                <div class="file-info" id="fileInfo" style="display:none;">
                    <span class="file-name" id="fileName"></span>
                    <button class="btn-remove" id="removeFile">&#x2715;</button>
                </div>
            </div>
            <div class="query-section">
                <label for="queryInput">What would you like to know?</label>
                <textarea id="queryInput" rows="3"
                    placeholder="e.g., What is the distribution of sales by region?"></textarea>
                <div class="example-queries">
                    <p>Example questions:</p>
                    <div class="query-chips">
                        <button class="query-chip" data-query="Provide a comprehensive exploratory data analysis">&#128202; Full EDA</button>
                        <button class="query-chip" data-query="What are the key patterns and correlations in this data?">&#128269; Find Patterns</button>
                        <button class="query-chip" data-query="Show me the distribution of values and identify any outliers">&#128200; Distributions</button>
                        <button class="query-chip" data-query="Compare different categories and identify top performers">&#127942; Compare Categories</button>
                    </div>
                </div>
            </div>
            <button class="btn-primary" id="analyzeBtn" disabled>
                <span class="btn-text">Analyze Dataset</span>
                <span class="btn-loading" style="display:none;">
                    <span class="spinner"></span> Analyzing...
                </span>
            </button>
            <div class="change-key-row">
                <button class="btn-link" id="changeKeyBtn">&#128273; Change API Key</button>
            </div>
        </section>

        <!-- Results Section -->
        <section class="results-section" id="resultsSection" style="display:none;">
            <div class="results-header">
                <h2>Analysis Results</h2>
                <button class="btn-secondary" id="newAnalysisBtn">&#128228; New Analysis</button>
            </div>
            <div class="results-grid">
                <div class="result-card summary-card">
                    <div class="card-header">
                        <h3>&#128161; Summary</h3>
                        <span class="quality-badge" id="qualityBadge"></span>
                    </div>
                    <div class="card-content">
                        <div id="summaryText" class="summary-text"></div>
                        <div class="metrics">
                            <div class="metric"><span class="metric-label">Analysis Time</span><span class="metric-value" id="latencyValue">-</span></div>
                            <div class="metric"><span class="metric-label">Steps Executed</span><span class="metric-value" id="stepsValue">-</span></div>
                            <div class="metric"><span class="metric-label">Visualizations</span><span class="metric-value" id="vizValue">-</span></div>
                        </div>
                    </div>
                </div>
                <div class="result-card">
                    <div class="card-header"><h3>&#128202; Visualizations</h3></div>
                    <div class="card-content"><div id="visualizationsContainer" class="visualizations-grid"></div></div>
                </div>
                <div class="result-card">
                    <div class="card-header"><h3>&#128269; Analysis Steps</h3></div>
                    <div class="card-content"><div id="stepsContainer" class="steps-list"></div></div>
                </div>
            </div>
            <div class="download-section">
                <h3>&#128229; Download Results</h3>
                <div class="download-buttons">
                    <button class="btn-download" id="downloadArtifactBtn"><span>&#128196; Full Report (JSON)</span></button>
                    <button class="btn-download" id="downloadVisualizationsBtn"><span>&#128444; All Visualizations</span></button>
                </div>
            </div>
        </section>

        <!-- Error Section -->
        <section class="error-section" id="errorSection" style="display:none;">
            <div class="error-card">
                <div class="error-icon">&#9888;</div>
                <h3>Analysis Failed</h3>
                <p id="errorMessage"></p>
                <button class="btn-secondary" id="tryAgainBtn">Try Again</button>
            </div>
        </section>
    </main>
    <footer class="footer">
        <p>Data Analysis Agent &mdash; CS539 Project</p>
        <p>Hongchao Hu &amp; Niyati Gohil</p>
    </footer>
</div>
<script src="/static/js/app.js"></script>
</body>
</html>"""

(ROOT / "static/index.html").write_text(html)
print("✅ index.html written")

In [ ]:
# ── Cell 8: Write CSS (extends original) ──────────────────────────────────────

css_extra = r"""
/* API Key Section */
.apikey-section {
    max-width: 600px;
    margin: 0 auto 30px;
}
.apikey-card {
    background: white;
    border: 2px solid var(--border-color);
    border-radius: var(--border-radius);
    padding: 30px;
    box-shadow: var(--shadow-sm);
}
.apikey-card h2 {
    margin-bottom: 10px;
}
.apikey-card p {
    color: var(--text-secondary);
    margin-bottom: 16px;
    font-size: 0.95rem;
}
.apikey-card a { color: var(--primary-color); }
.apikey-input-row {
    display: flex;
    gap: 10px;
}
.apikey-input-row input {
    flex: 1;
    padding: 10px 14px;
    border: 2px solid var(--border-color);
    border-radius: 8px;
    font-size: 1rem;
    font-family: monospace;
}
.apikey-input-row input:focus {
    outline: none;
    border-color: var(--primary-color);
}
.apikey-note {
    font-size: 0.8rem !important;
    color: var(--text-tertiary) !important;
    margin-top: 10px !important;
    margin-bottom: 0 !important;
}
.change-key-row {
    text-align: center;
    margin-top: 10px;
}
.btn-link {
    background: none;
    border: none;
    color: var(--text-secondary);
    font-size: 0.85rem;
    cursor: pointer;
    text-decoration: underline;
}
.btn-link:hover { color: var(--primary-color); }

/* Markdown-rendered summary */
.summary-text h1, .summary-text h2, .summary-text h3 {
    margin: 12px 0 6px;
    color: var(--text-primary);
}
.summary-text p { margin-bottom: 10px; }
.summary-text ul, .summary-text ol {
    padding-left: 20px;
    margin-bottom: 10px;
}
.summary-text li { margin-bottom: 4px; }
.summary-text strong { font-weight: 700; }
"""

# Read original CSS from the uploaded project or use a minimal version
original_css_path = Path("/content/data_analysis_agent/static/css/styles.css")
if not original_css_path.exists():
    # Write the full CSS inline
    pass  # will be written below

# Append extra styles to existing CSS file
with open(ROOT / "static/css/styles.css", "a") as f:
    f.write(css_extra)

print("✅ CSS updated")

In [ ]:
# ── Cell 9: Write JavaScript ───────────────────────────────────────────────────

js = r"""
// ── Config ──────────────────────────────────────────────────────────────────
const API_BASE_URL = window.location.origin;
let geminiApiKey = '';
let selectedFile = null;
let currentArtifactId = null;
let currentVisualizations = [];

// ── DOM ──────────────────────────────────────────────────────────────────────
const el = (id) => document.getElementById(id);

// ── Init ─────────────────────────────────────────────────────────────────────
document.addEventListener('DOMContentLoaded', () => {
    checkHealth();
    setupApiKey();
    setupUpload();
    setupActions();
    el('queryInput').value = 'Provide a comprehensive exploratory data analysis with key insights, patterns, and visualizations.';
});

// ── Health ───────────────────────────────────────────────────────────────────
async function checkHealth() {
    try {
        const r = await fetch(`${API_BASE_URL}/health`);
        const d = await r.json();
        if (d.status === 'healthy') {
            el('statusDot').classList.add('healthy');
            el('statusText').textContent = 'API Ready';
        } else {
            el('statusText').textContent = 'API Unavailable';
        }
    } catch { el('statusText').textContent = 'Connecting...'; }
}

// ── API Key ───────────────────────────────────────────────────────────────────
function setupApiKey() {
    el('saveKeyBtn').addEventListener('click', () => {
        const key = el('apiKeyInput').value.trim();
        if (!key) { alert('Please enter your Gemini API key.'); return; }
        geminiApiKey = key;
        el('apikeySection').style.display = 'none';
        el('uploadSection').style.display = 'block';
    });
    el('apiKeyInput').addEventListener('keydown', (e) => { if (e.key === 'Enter') el('saveKeyBtn').click(); });
}

// ── Upload ────────────────────────────────────────────────────────────────────
function setupUpload() {
    const area = el('uploadArea');
    area.addEventListener('click', (e) => { if (!e.target.closest('.btn-remove')) el('fileInput').click(); });
    el('fileInput').addEventListener('change', (e) => { if (e.target.files[0]) handleFile(e.target.files[0]); });
    el('removeFile').addEventListener('click', (e) => { e.stopPropagation(); clearFile(); });

    ['dragenter','dragover','dragleave','drop'].forEach(ev => area.addEventListener(ev, (e) => { e.preventDefault(); e.stopPropagation(); }));
    ['dragenter','dragover'].forEach(ev => area.addEventListener(ev, () => area.classList.add('dragover')));
    ['dragleave','drop'].forEach(ev => area.addEventListener(ev, () => area.classList.remove('dragover')));
    area.addEventListener('drop', (e) => { const f = e.dataTransfer.files[0]; if (f) handleFile(f); });
}

function handleFile(file) {
    if (!file.name.toLowerCase().endsWith('.csv')) { alert('Please upload a CSV file.'); return; }
    if (file.size > 50 * 1024 * 1024) { alert('File exceeds 50 MB limit.'); return; }
    selectedFile = file;
    el('fileName').textContent = file.name;
    el('fileInfo').style.display = 'flex';
    el('analyzeBtn').disabled = false;
    el('uploadArea').querySelector('h2').textContent = 'File selected!';
    el('uploadArea').querySelector('p').textContent = 'Click "Analyze Dataset" to continue';
}

function clearFile() {
    selectedFile = null;
    el('fileInput').value = '';
    el('fileInfo').style.display = 'none';
    el('analyzeBtn').disabled = true;
    el('uploadArea').querySelector('h2').textContent = 'Drop your CSV file here';
    el('uploadArea').querySelector('p').textContent = 'or click to browse';
}

// ── Actions ───────────────────────────────────────────────────────────────────
function setupActions() {
    el('analyzeBtn').addEventListener('click', handleAnalyze);
    el('newAnalysisBtn').addEventListener('click', () => showSection('upload'));
    el('tryAgainBtn').addEventListener('click', () => showSection('upload'));
    el('changeKeyBtn').addEventListener('click', () => {
        geminiApiKey = '';
        showSection('apikey');
    });
    el('downloadArtifactBtn').addEventListener('click', downloadArtifact);
    el('downloadVisualizationsBtn').addEventListener('click', downloadVisualizations);
    document.querySelectorAll('.query-chip').forEach(c => {
        c.addEventListener('click', () => { el('queryInput').value = c.dataset.query; });
    });
}

function showSection(name) {
    el('apikeySection').style.display = name === 'apikey' ? 'block' : 'none';
    el('uploadSection').style.display = name === 'upload' ? 'block' : 'none';
    el('resultsSection').style.display = name === 'results' ? 'block' : 'none';
    el('errorSection').style.display = name === 'error' ? 'block' : 'none';
    if (name === 'upload') clearFile();
}

// ── Analyze ───────────────────────────────────────────────────────────────────
async function handleAnalyze() {
    if (!selectedFile || !el('queryInput').value.trim()) return;

    const btn = el('analyzeBtn');
    btn.disabled = true;
    el('analyzeBtn').querySelector('.btn-text').style.display = 'none';
    el('analyzeBtn').querySelector('.btn-loading').style.display = 'flex';

    const form = new FormData();
    form.append('file', selectedFile);
    form.append('query', el('queryInput').value.trim());

    try {
        const response = await fetch(`${API_BASE_URL}/upload-analyze`, {
            method: 'POST',
            headers: { 'X-API-Key': geminiApiKey },
            body: form
        });
        if (!response.ok) {
            const err = await response.json();
            throw new Error(err.detail || 'Analysis failed');
        }
        const result = await response.json();
        displayResults(result);
    } catch (e) {
        el('errorMessage').textContent = e.message || 'An error occurred.';
        showSection('error');
        btn.disabled = false;
        el('analyzeBtn').querySelector('.btn-text').style.display = 'inline';
        el('analyzeBtn').querySelector('.btn-loading').style.display = 'none';
    }
}

// ── Display ───────────────────────────────────────────────────────────────────
function displayResults(data) {
    currentArtifactId = data.artifact_id;
    currentVisualizations = data.visualizations || [];

    // Render markdown summary
    el('summaryText').innerHTML = markdownToHtml(data.summary || '');

    el('latencyValue').textContent = `${Number(data.latency).toFixed(2)}s`;
    el('stepsValue').textContent = data.steps_executed ?? '-';
    el('vizValue').textContent = currentVisualizations.length;
    el('qualityBadge').style.display = 'none';

    // Visualizations
    const vc = el('visualizationsContainer');
    vc.innerHTML = '';
    if (currentVisualizations.length === 0) {
        vc.innerHTML = '<p style="color:var(--text-secondary)">No visualizations generated.</p>';
    } else {
        currentVisualizations.forEach((viz, i) => {
            const item = document.createElement('div');
            item.className = 'viz-item';
            const img = document.createElement('img');
            img.src = `${API_BASE_URL}/visualization/${viz}`;
            img.alt = `Visualization ${i + 1}`;
            img.onerror = () => { img.alt = 'Image not available'; };
            const lbl = document.createElement('div');
            lbl.className = 'viz-label';
            lbl.textContent = viz.replace(/\.(png|jpg)$/i,'').replace(/_\d{8}_\d{6}$/,'').replace(/_/g,' ');
            item.append(img, lbl);
            vc.appendChild(item);
        });
    }

    // Steps
    const sc = el('stepsContainer');
    sc.innerHTML = '';
    (data.analysis_plan || []).forEach((s, i) => {
        const d = document.createElement('div');
        d.className = 'step-item';
        d.innerHTML = `<h4>${i + 1}. ${s.tool || ''}</h4><p>${s.purpose || ''}</p>`;
        sc.appendChild(d);
    });

    showSection('results');
}

// Simple markdown → HTML (bold, headings, lists)
function markdownToHtml(md) {
    return md
        .replace(/&/g,'&amp;').replace(/</g,'&lt;').replace(/>/g,'&gt;')
        .replace(/^######\s(.+)$/gm,'<h6>$1</h6>')
        .replace(/^#####\s(.+)$/gm,'<h5>$1</h5>')
        .replace(/^####\s(.+)$/gm,'<h4>$1</h4>')
        .replace(/^###\s(.+)$/gm,'<h3>$1</h3>')
        .replace(/^##\s(.+)$/gm,'<h2>$1</h2>')
        .replace(/^#\s(.+)$/gm,'<h1>$1</h1>')
        .replace(/\*\*(.+?)\*\*/g,'<strong>$1</strong>')
        .replace(/\*(.+?)\*/g,'<em>$1</em>')
        .replace(/^[-*]\s(.+)$/gm,'<li>$1</li>')
        .replace(/^\d+\.\s(.+)$/gm,'<li>$1</li>')
        .replace(/(<li>.*<\/li>)/s,'<ul>$1</ul>')
        .replace(/\n\n/g,'</p><p>')
        .replace(/^(?!<[h|u|o|l])(.+)$/gm,'<p>$1</p>')
        .replace(/<p><\/p>/g,'');
}

// ── Downloads ─────────────────────────────────────────────────────────────────
async function downloadArtifact() {
    if (!currentArtifactId) return;
    const r = await fetch(`${API_BASE_URL}/artifact/${currentArtifactId}`);
    const data = await r.json();
    const a = Object.assign(document.createElement('a'), {
        href: URL.createObjectURL(new Blob([JSON.stringify(data,null,2)],{type:'application/json'})),
        download: `analysis_${currentArtifactId}.json`
    });
    a.click(); URL.revokeObjectURL(a.href);
}

async function downloadVisualizations() {
    for (const viz of currentVisualizations) {
        const r = await fetch(`${API_BASE_URL}/visualization/${viz}`);
        const blob = await r.blob();
        const a = Object.assign(document.createElement('a'), {
            href: URL.createObjectURL(blob), download: viz
        });
        a.click(); URL.revokeObjectURL(a.href);
        await new Promise(res => setTimeout(res, 200));
    }
}
"""

(ROOT / "static/js/app.js").write_text(js)
print("✅ app.js written")

In [ ]:
# ── Cell 10: Copy CSS from original styles (base styles) ──────────────────────
# This cell writes the base CSS that styles.css.backup may be missing.
# We write a full standalone CSS so the Colab version is self-contained.

base_css = r"""
:root {
    --primary-color: #333333;
    --primary-hover: #1a1a1a;
    --secondary-color: #666666;
    --danger-color: #444444;
    --warning-color: #888888;
    --success-color: #22c55e;
    --bg-primary: #ffffff;
    --bg-secondary: #f9fafb;
    --bg-tertiary: #f3f4f6;
    --text-primary: #111827;
    --text-secondary: #6b7280;
    --text-tertiary: #9ca3af;
    --border-color: #e5e7eb;
    --border-radius: 12px;
    --shadow-sm: 0 1px 2px 0 rgba(0,0,0,.05);
    --shadow-md: 0 4px 6px -1px rgba(0,0,0,.1);
    --shadow-lg: 0 10px 15px -3px rgba(0,0,0,.1);
}
* { margin:0; padding:0; box-sizing:border-box; }
body { font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,'Helvetica Neue',Arial,sans-serif; line-height:1.6; color:var(--text-primary); background:linear-gradient(135deg,#2a2a2a 0%,#1a1a1a 100%); min-height:100vh; padding:20px; }
.container { max-width:1200px; margin:0 auto; background:var(--bg-primary); border-radius:var(--border-radius); box-shadow:var(--shadow-lg); overflow:hidden; }
.header { background:linear-gradient(135deg,#2a2a2a 0%,#1a1a1a 100%); color:white; padding:40px 30px; text-align:center; }
.header h1 { font-size:2.5rem; margin-bottom:10px; font-weight:700; }
.subtitle { font-size:1.1rem; opacity:.9; margin-bottom:20px; }
.status-indicator { display:inline-flex; align-items:center; gap:8px; background:rgba(0,0,0,.2); padding:8px 16px; border-radius:20px; font-size:.9rem; }
.status-dot { width:10px; height:10px; border-radius:50%; background:#d1d5db; }
.status-dot.healthy { background:var(--success-color); animation:pulse 2s infinite; }
@keyframes pulse { 0%,100%{opacity:1} 50%{opacity:.5} }
.main-content { padding:40px 30px; }
.upload-section { max-width:800px; margin:0 auto; }
.upload-area { border:3px dashed var(--border-color); border-radius:var(--border-radius); padding:60px 40px; text-align:center; background:var(--bg-secondary); transition:all .3s; cursor:pointer; position:relative; }
.upload-area:hover { border-color:var(--primary-color); background:var(--bg-tertiary); }
.upload-area.dragover { border-color:var(--primary-color); background:rgba(51,51,51,.1); transform:scale(1.02); }
.upload-icon { font-size:4rem; margin-bottom:20px; }
.upload-area h2 { font-size:1.5rem; margin-bottom:10px; }
.upload-area p { color:var(--text-secondary); }
.file-info { margin-top:20px; display:flex; align-items:center; justify-content:center; gap:10px; background:white; padding:15px 20px; border-radius:8px; border:1px solid var(--border-color); }
.file-name { font-weight:600; color:var(--primary-color); }
.btn-remove { background:var(--danger-color); color:white; border:none; border-radius:50%; width:24px; height:24px; cursor:pointer; display:flex; align-items:center; justify-content:center; font-size:1rem; transition:all .2s; }
.btn-remove:hover { transform:scale(1.1); }
.query-section { margin-top:30px; }
.query-section label { display:block; font-weight:600; margin-bottom:10px; }
.query-section textarea { width:100%; padding:15px; border:2px solid var(--border-color); border-radius:8px; font-family:inherit; font-size:1rem; resize:vertical; transition:border-color .3s; }
.query-section textarea:focus { outline:none; border-color:var(--primary-color); }
.example-queries { margin-top:15px; }
.example-queries p { font-size:.9rem; color:var(--text-secondary); margin-bottom:8px; }
.query-chips { display:flex; flex-wrap:wrap; gap:10px; }
.query-chip { padding:8px 16px; background:var(--bg-tertiary); border:1px solid var(--border-color); border-radius:20px; font-size:.85rem; cursor:pointer; transition:all .2s; }
.query-chip:hover { background:var(--primary-color); color:white; border-color:var(--primary-color); }
.btn-primary { width:100%; padding:16px 32px; background:var(--primary-color); color:white; border:none; border-radius:8px; font-size:1.1rem; font-weight:600; cursor:pointer; margin-top:30px; transition:all .3s; position:relative; }
.btn-primary:hover:not(:disabled) { background:var(--primary-hover); transform:translateY(-2px); box-shadow:var(--shadow-md); }
.btn-primary:disabled { background:var(--text-tertiary); cursor:not-allowed; }
.btn-loading { display:flex; align-items:center; justify-content:center; gap:10px; }
.spinner { width:20px; height:20px; border:3px solid rgba(255,255,255,.3); border-top-color:white; border-radius:50%; animation:spin 1s linear infinite; }
@keyframes spin { to{transform:rotate(360deg)} }
.btn-secondary { padding:10px 20px; background:white; color:var(--primary-color); border:2px solid var(--primary-color); border-radius:8px; font-weight:600; cursor:pointer; transition:all .2s; }
.btn-secondary:hover { background:var(--primary-color); color:white; }
.btn-download { padding:12px 24px; background:var(--bg-tertiary); border:2px solid var(--border-color); border-radius:8px; cursor:pointer; transition:all .2s; display:flex; align-items:center; gap:8px; }
.btn-download:hover { background:var(--primary-color); color:white; border-color:var(--primary-color); }
.results-section { animation:fadeIn .5s ease; }
@keyframes fadeIn { from{opacity:0;transform:translateY(20px)} to{opacity:1;transform:none} }
.results-header { display:flex; justify-content:space-between; align-items:center; margin-bottom:30px; }
.results-header h2 { font-size:2rem; }
.results-grid { display:grid; gap:20px; }
.result-card { background:white; border:1px solid var(--border-color); border-radius:var(--border-radius); padding:25px; box-shadow:var(--shadow-sm); }
.card-header { display:flex; justify-content:space-between; align-items:center; margin-bottom:20px; padding-bottom:15px; border-bottom:2px solid var(--bg-tertiary); }
.card-header h3 { font-size:1.3rem; }
.quality-badge { padding:6px 16px; border-radius:20px; font-size:.85rem; font-weight:600; }
.summary-text { white-space:normal; line-height:1.8; color:var(--text-primary); margin-bottom:20px; font-size:1rem; }
.metrics { display:grid; grid-template-columns:repeat(auto-fit,minmax(150px,1fr)); gap:15px; margin-top:20px; }
.metric { background:var(--bg-secondary); padding:15px; border-radius:8px; text-align:center; }
.metric-label { display:block; font-size:.85rem; color:var(--text-secondary); margin-bottom:5px; }
.metric-value { display:block; font-size:1.5rem; font-weight:700; color:var(--primary-color); }
.visualizations-grid { display:grid; grid-template-columns:repeat(auto-fit,minmax(300px,1fr)); gap:20px; }
.viz-item { border:1px solid var(--border-color); border-radius:8px; overflow:hidden; transition:transform .2s; }
.viz-item:hover { transform:scale(1.02); }
.viz-item img { width:100%; height:auto; display:block; }
.viz-label { padding:10px; background:var(--bg-tertiary); font-size:.9rem; color:var(--text-secondary); text-align:center; }
.steps-list { display:flex; flex-direction:column; gap:15px; }
.step-item { padding:15px; background:var(--bg-secondary); border-left:4px solid var(--primary-color); border-radius:4px; }
.step-item h4 { font-size:1rem; margin-bottom:5px; }
.step-item p { font-size:.9rem; color:var(--text-secondary); }
.download-section { margin-top:30px; padding:25px; background:var(--bg-secondary); border-radius:var(--border-radius); }
.download-section h3 { margin-bottom:15px; }
.download-buttons { display:flex; gap:15px; flex-wrap:wrap; }
.error-section { max-width:600px; margin:0 auto; }
.error-card { background:white; border:2px solid var(--danger-color); border-radius:var(--border-radius); padding:40px; text-align:center; }
.error-icon { font-size:4rem; margin-bottom:20px; }
.error-card h3 { color:var(--danger-color); margin-bottom:15px; }
.error-card p { color:var(--text-secondary); margin-bottom:25px; }
.footer { background:var(--bg-tertiary); padding:20px 30px; text-align:center; color:var(--text-secondary); font-size:.9rem; border-top:1px solid var(--border-color); }
.footer p { margin:5px 0; }
@media(max-width:768px) {
    body{padding:10px}
    .header h1{font-size:2rem}
    .main-content{padding:20px 15px}
    .upload-area{padding:40px 20px}
    .query-chips{flex-direction:column}
    .results-header{flex-direction:column;gap:15px;align-items:flex-start}
    .metrics{grid-template-columns:1fr}
    .download-buttons{flex-direction:column}
}
"""

# Write base CSS first, then append the extra styles
(ROOT / "static/css/styles.css").write_text(base_css)

css_extra = r"""
.apikey-section { max-width:600px; margin:0 auto 30px; }
.apikey-card { background:white; border:2px solid var(--border-color); border-radius:var(--border-radius); padding:30px; box-shadow:var(--shadow-sm); }
.apikey-card h2 { margin-bottom:10px; }
.apikey-card p { color:var(--text-secondary); margin-bottom:16px; font-size:.95rem; }
.apikey-card a { color:var(--primary-color); }
.apikey-input-row { display:flex; gap:10px; }
.apikey-input-row input { flex:1; padding:10px 14px; border:2px solid var(--border-color); border-radius:8px; font-size:1rem; font-family:monospace; }
.apikey-input-row input:focus { outline:none; border-color:var(--primary-color); }
.apikey-note { font-size:.8rem !important; color:var(--text-tertiary) !important; margin-top:10px !important; margin-bottom:0 !important; }
.change-key-row { text-align:center; margin-top:10px; }
.btn-link { background:none; border:none; color:var(--text-secondary); font-size:.85rem; cursor:pointer; text-decoration:underline; }
.btn-link:hover { color:var(--primary-color); }
.summary-text h1,.summary-text h2,.summary-text h3 { margin:12px 0 6px; color:var(--text-primary); }
.summary-text p { margin-bottom:10px; }
.summary-text ul,.summary-text ol { padding-left:20px; margin-bottom:10px; }
.summary-text li { margin-bottom:4px; }
.summary-text strong { font-weight:700; }
"""

with open(ROOT / "static/css/styles.css", "a") as f:
    f.write(css_extra)

print("✅ CSS written")

In [ ]:
# ── Cell 11: Start FastAPI server (background) ────────────────────────────────

import subprocess, time, requests

server_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=str(ROOT)
)

# Wait until server is ready
for _ in range(30):
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            print("✅ Server is running on http://localhost:8000")
            break
    except Exception:
        pass
    time.sleep(2)
else:
    print("⚠️  Server may not have started — check output above")

In [ ]:
# ── Cell 12: Open public tunnel with cloudflared (no sign-in needed) ──────────

import subprocess, threading, re, time
from IPython.display import display, HTML

# Download cloudflared binary (one-time)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

public_url = None

def run_tunnel():
    global public_url
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
        if match:
            public_url = match.group(0)
            break

t = threading.Thread(target=run_tunnel, daemon=True)
t.start()

# Wait for URL
for _ in range(30):
    if public_url:
        break
    time.sleep(2)

if public_url:
    display(HTML(f"""
    <div style="font-family:sans-serif;background:#f0fdf4;border:2px solid #22c55e;
                border-radius:12px;padding:24px;max-width:600px;margin:20px auto">
        <h2 style="color:#166534;margin-bottom:8px">&#127881; Your app is live!</h2>
        <p style="color:#166534;margin-bottom:16px">Share this link with anyone — no account or installation needed:</p>
        <a href="{public_url}" target="_blank"
           style="display:block;background:#166534;color:white;padding:14px 20px;
                  border-radius:8px;text-decoration:none;font-size:1.1rem;text-align:center;word-break:break-all">
            &#128279; {public_url}
        </a>
        <p style="color:#6b7280;font-size:.85rem;margin-top:12px">
            &#9888; This URL is temporary and changes when you restart the notebook.
            Keep this Colab tab open while the app is in use.
        </p>
    </div>
    """))
else:
    print("⚠️  Could not get a public URL. Try running this cell again.")